# 实验 21 — dbt hooks（pre/post + on-run-start/end）

**目标**：在模型物化前后、整个 run 前后插入 SQL。生产中最常见用途：

- **grants**：每次建表后 grant 给 BI 角色
- **audit log**：on-run-end 写一条记录到 audit 表（run_id、运行时长、失败模型）
- **vacuum / optimize**：建完表后跑维护操作（Snowflake `ALTER TABLE SUSPEND CLUSTERING`、Postgres `VACUUM`）
- **session 参数**：on-run-start 设 `set query_tag='dbt'`、时区、search_path

四种 hook 时序：

```
on-run-start  ← 整个 run 最前
  ┌─ model A:  pre-hook → SQL → post-hook
  ├─ model B:  pre-hook → SQL → post-hook
  └─ ...
on-run-end    ← 整个 run 最后
```

In [ ]:
# 1) 临时改 dbt_project.yml 加 hooks 配置
from pathlib import Path
p = Path('../dbt_project/dbt_project.yml')
orig = p.read_text()
p.write_text(orig.replace(
    'models:\n  dlt_dbt_sandbox:',
    '''on-run-start:
  - "create schema if not exists audit"
  - "create table if not exists audit.dbt_runs (invocation_id varchar, started_at timestamp, status varchar, ended_at timestamp)"
  - "insert into audit.dbt_runs values ('{{ invocation_id }}', current_timestamp, 'started', null)"

on-run-end:
  - "update audit.dbt_runs set status = '{{ \"failed\" if results | selectattr(\"status\", \"equalto\", \"error\") | list | length > 0 else \"success\" }}', ended_at = current_timestamp where invocation_id = '{{ invocation_id }}' and ended_at is null"

models:
  dlt_dbt_sandbox:'''
))
print(p.read_text()[:1000])

In [ ]:
# 2) 给 fct_daily_rates 加 post-hook：建完表后注释一下
# 改 _marts.yml 太显眼；这里用 in-line config
model = Path('../dbt_project/models/marts/fct_daily_rates.sql')
morig = model.read_text()
model.write_text("{{ config(post_hook=\"comment on table {{ this }} is 'rebuilt by dbt run-id {{ invocation_id }}'\") }}\n\n" + morig)
print(model.read_text()[:400])

In [ ]:
import subprocess, os
r = subprocess.run(['uv','run','dbt','build','--exclude','snp_currencies'],
                   capture_output=True, text=True, cwd='../dbt_project',
                   env={**os.environ,'DBT_PROFILES_DIR':'.'})
print(r.stdout[-2500:])

In [ ]:
# 看 audit 表，应该有两条记录（started → success）
import duckdb
con = duckdb.connect('../data/sandbox.duckdb')
con.execute('select * from audit.dbt_runs order by started_at desc limit 5').df()

In [ ]:
# 验证 post-hook：表注释
row = con.execute("""
    select comment from duckdb_tables() 
    where schema_name='main' and table_name='fct_daily_rates'
""").fetchone()
print('table comment:', row)

In [ ]:
# 还原配置
Path('../dbt_project/dbt_project.yml').write_text(orig)
Path('../dbt_project/models/marts/fct_daily_rates.sql').write_text(morig)

## 生产环境踩坑提示

- **hook 顺序敏感**：pre-hook 失败模型不建。on-run-start 失败整个 run 直接挂。on-run-end 不管前面成败都跑。
- **on-run-end 拿到 `results`** —— 一个对象列表，能区分 status / message / execution_time，写监控告警时很有用。
- **不要在 hook 里跑“另一个模型”**。如果你想在 marts 之后建一些“非 dbt-managed” 的对象，考虑用 macro + `--operation` 而不是 hook。
- **多 target 区分**：on-run-end 写 audit 表要避免 dev 写到 prod 的 audit。常规做法是 `{{ generate_schema_name() }}` 给 audit 表也加 prefix。
- **grants 用 `grants` config 更好**：dbt 1.5+ 的 `grants` 比 post-hook GRANT 更声明式：
  ```yaml
  models:
    +grants:
      select: ['bi_role']
  ```

## 思考题

- 把 audit 改成结构化失败明细：on-run-end 里遍历 `results`，把每个 failed 模型的 unique_id、message 单独写一行。
- 用 `query_tag`（Snowflake）/ `set application_name`（Postgres）的 pre-hook，在数据库慢查询日志里能看到 “这是 dbt 跑的”。
- dbt 1.5+ 的 `grants` config 比 post-hook GRANT 强在哪里？（提示：声明式 + 自动 revoke 多余权限）